# EDA

Run `Data_Cleaning.ipynb` first to create the cleaned dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
import os
os.chdir(PROJECT_ROOT)


In [ ]:
data = pd.read_csv("data/processed/dataset_Cleaned.csv")
print("Cleaned dataset shape:", data.shape)

Explore the cleaned data to understand the target outcome and how the credit-bureau features relate to it.

### Outcome Distribution

Locate the OUTCOME column and see how many records fall into each outcome category.

In [ ]:
[col for col in data.columns if "OU" in col]

In [ ]:
data["OUTCOME"].value_counts()

### Remove Policy-Declined Records

Drop the policy-declined applications, since they were never scored on merit and would distort the analysis.

In [ ]:
data = data[data["OUTCOME"] != "Rejected - Policy Decline"]


### Credit Scores (CR21 vs CR22) by Outcome

Compare the two credit raters' scores across outcome groups using side-by-side boxplots.

In [ ]:
print(data[['RISK_SCORE', 'NO_SCORE', "SCORE_CR22"]])

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(data=data, x="OUTCOME", y="NO_SCORE", ax=axes[0])
axes[0].set_title("CR_21 SCORES by TARGET", fontsize=16)
axes[0].set_xlabel("TARGET", fontsize=15)
axes[0].set_ylabel("RISK_SCORE", fontsize=15)
axes[0].tick_params(labelsize=12)

sns.boxplot(data=data, x="OUTCOME", y="SCORE_CR22", ax=axes[1])
axes[1].set_title("CR_22 SCORES  by TARGET", fontsize=16)
axes[1].set_xlabel("TARGET", fontsize=15)
axes[1].set_ylabel("SCORE_CR22", fontsize=15)
axes[1].tick_params(labelsize=12)

plt.tight_layout()
plt.show()

### Misclassification Check

Count the records where a score disagrees with the actual outcome — good applicants with low scores, or bad applicants with high scores.

In [ ]:
mask = (data["OUTCOME"] == "Good") & (data["NO_SCORE"] < 460)
print("Wrongly clasified Bad as Good (By CR21):", mask.sum())

In [ ]:
mask = (data["OUTCOME"] == "Bad") & (data["SCORE_CR22"] > 870)
print("Wrongly clasified Bad as Good (By CR22):", mask.sum())

In [ ]:
data.info()

### Risk Relative Index Distribution

Compare the CR21 risk-relative index for Good versus Bad outcomes using overlaid density curves.

In [ ]:
col = "RISK_RELATIVE_CR21"

plt.figure(figsize=(8, 6))
sns.kdeplot(data[data["OUTCOME"] == "Good"][col], fill=True, alpha=0.5,
            color="skyblue", label="Good")
sns.kdeplot(data[data["OUTCOME"] == "Bad"][col], fill=True, alpha=0.5,
            color="orange", label="Bad")
plt.title("Risk Relative Index by TARGET")
plt.xlabel("Risk Relative Index")
plt.ylabel("Density")
plt.legend(loc="upper left")
plt.xlim(-100, 100)      # <-- x-axis range
plt.tight_layout()
plt.show()

In [ ]:
col = "RISK_RELATIVE_CR21"

plt.figure(figsize=(8, 6))
sns.kdeplot(data[data["OUTCOME"] == "Good"][col], fill=True, alpha=0.5,
            color="skyblue", label="Good")
sns.kdeplot(data[data["OUTCOME"] == "Bad"][col], fill=True, alpha=0.5,
            color="orange", label="Bad")
plt.title("Risk Relative Index by TARGET")
plt.xlabel("Risk Relative Index")
plt.ylabel("Density")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

### Employment Status vs Outcome

Check whether employment status is associated with the outcome using a proportional stacked bar chart.

In [ ]:
import matplotlib.pyplot as plt

ct = pd.crosstab(data["EMPLOYED_STATUS"], data["OUTCOME"], normalize="index")
ct.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.title("OUTCOME by EMPLOYED_STATUS")
plt.ylabel("Proportion")
plt.tight_layout()
plt.show()

### Worst Delinquency Status by Outcome

Compare the worst delinquency status over different time windows across outcome groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

worst_cols = ["WORST_1M", "WORST_3M", "WORST_6M", "WORST_12M", "WORST_18M", "WORST_24M"]
group = "OUTCOME"     # your Good/Bad column (or use "OUTCOME")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flatten(), worst_cols):
    sns.boxplot(data=data, x=group, y=col, ax=ax)
    ax.set_title(f"{col} by {group}")
plt.tight_layout()
plt.show()

### Worst Delinquency vs Credit Scores

See how the worst delinquency status relates to the two credit scores by plotting the average score at each delinquency level.

In [ ]:
score_cols = ["SCORE_CR22", "NO_SCORE"]

fig, axes = plt.subplots(len(score_cols), len(worst_cols),
                         figsize=(22, 8), sharey="row")

for r, score in enumerate(score_cols):
    for c, worst in enumerate(worst_cols):
        means = data.groupby(worst)[score].mean()
        axes[r, c].plot(means.index, means.values, marker="o")
        axes[r, c].set_title(f"{score} vs {worst}", fontsize=9)

plt.tight_layout()
plt.show()

### Risk Columns Inspection

Identify the risk-related columns and review the summary statistics of the CR21 risk-relative index.

In [ ]:
[col for col in data.columns if "RISK" in col]

In [ ]:

data[["RISK_RELATIVE_CR21" ]].describe()